# 03. EDA, Acoustic Indicators and Emotion Timeline

**Project:** Audio-Based Speaking Feedback System using Emotion Recognition and Acoustic Stress Indicators  
**Student:** Nguyễn Minh Cường  
**Role:** EDA, acoustic indicators, emotion timeline, visualization and feedback analysis  

## Purpose of this notebook

This notebook continues from the processed artifacts generated in the previous data preparation stage.  
Instead of parsing and preprocessing the raw datasets again, this notebook loads the processed metadata, baseline feature files, normalized audio, and Log-Mel tensors from the `ser_processed` folder.

The main goals are:

1. Analyze the structure and distribution of the processed speech emotion dataset.
2. Visualize important speech representations such as waveform, spectrogram, MFCC and Log-Mel spectrogram.
3. Analyze acoustic speaking indicators such as RMS energy, pitch/F0, ZCR, pause ratio and speaking continuity.
4. Build a simple emotion timeline framework for long speech audio.
5. Prepare figures and explanations for the midterm report and presentation.

The acoustic indicators in this notebook are used only for speaking feedback and presentation practice.  
They are not used for medical diagnosis or clinical stress detection.

In [ ]:
# Optional Google Drive mount. This cell is skipped automatically outside Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print("Not running in Colab; using local project paths.")


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

import librosa
import librosa.display

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded successfully.")
print("librosa version:", librosa.__version__)

## 1. Loading processed artifacts

The previous stage already created a processed folder named `ser_processed`.  
This folder contains metadata, extracted baseline features, normalized 16 kHz audio files, Log-Mel tensors and the processing configuration.

In this notebook, we load these artifacts directly to avoid repeating the raw data parsing and preprocessing steps.

In [ ]:
def find_workspace_root():
    colab_root = Path('/content/drive/MyDrive/Speech_Project')
    if colab_root.exists():
        return colab_root
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, Path('/kaggle/working/Speech_Project'), Path('/kaggle/working')]
    for candidate in candidates:
        if (candidate / '01&02_Data_and_DataProcessing' / 'ser_processed').exists():
            return candidate
        if (candidate / 'Data_and_DataProcessing' / 'ser_processed').exists():
            return candidate
        if (candidate / 'ser_processed').exists():
            return candidate
    return cwd


WORKSPACE_ROOT = find_workspace_root()
if (WORKSPACE_ROOT / '01&02_Data_and_DataProcessing' / 'ser_processed').exists():
    INPUT_DIR = WORKSPACE_ROOT / '01&02_Data_and_DataProcessing' / 'ser_processed'
    OUTPUT_DIR = WORKSPACE_ROOT / '03_EDA' / 'ser_processed'
elif (WORKSPACE_ROOT / 'Data_and_DataProcessing' / 'ser_processed').exists():
    INPUT_DIR = WORKSPACE_ROOT / 'Data_and_DataProcessing' / 'ser_processed'
    OUTPUT_DIR = WORKSPACE_ROOT / 'EDA' / 'ser_processed'
else:
    INPUT_DIR = WORKSPACE_ROOT / 'ser_processed'
    OUTPUT_DIR = WORKSPACE_ROOT / '03_EDA' / 'ser_processed'

METADATA_PATH = INPUT_DIR / 'metadata.csv'
FEATURE_PATH = INPUT_DIR / 'baseline_features.npz'
CONFIG_PATH = INPUT_DIR / 'processing_config.json'

AUDIO_16K_DIR = INPUT_DIR / 'audio_16k'
LOG_MEL_DIR = INPUT_DIR / 'log_mel'
FIGURE_DIR = OUTPUT_DIR / 'figures_minhcuong'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print('WORKSPACE_ROOT:', WORKSPACE_ROOT)
print('INPUT_DIR:', INPUT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('FIGURE_DIR:', FIGURE_DIR)


In [ ]:
required_paths = {
    "metadata.csv": METADATA_PATH,
    "baseline_features.npz": FEATURE_PATH,
    "processing_config.json": CONFIG_PATH,
    "audio_16k folder": AUDIO_16K_DIR,
    "log_mel folder": LOG_MEL_DIR,
}

for name, path in required_paths.items():
    if path.exists():
        print(f"[OK] {name}: {path}")
    else:
        print(f"[MISSING] {name}: {path}")

In [ ]:
metadata = pd.read_csv(METADATA_PATH)
features = np.load(FEATURE_PATH, allow_pickle=True)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

print("Metadata shape:", metadata.shape)
print("Feature arrays:", features.files)
print("Processing config:")
print(json.dumps(config, indent=2))

In [ ]:
display(metadata.head())
display(metadata.info())

## 2. Checking processed audio paths

The previous notebook saved normalized audio files into the `audio_16k` folder.  
Each processed audio file is linked by `sample_id`.

For visualization and acoustic analysis, this notebook uses the processed 16 kHz audio when available.

In [ ]:
def get_processed_audio_path(sample_id):
    path = AUDIO_16K_DIR / f"{sample_id}.wav"
    return str(path) if path.exists() else None

metadata["processed_audio_path"] = metadata["sample_id"].apply(get_processed_audio_path)

available_audio = metadata["processed_audio_path"].notna().sum()
missing_audio = metadata["processed_audio_path"].isna().sum()

print("Available processed audio:", available_audio)
print("Missing processed audio:", missing_audio)

display(metadata[["sample_id", "dataset", "emotion", "split", "processed_audio_path"]].head())

## 3. Dataset overview

This section summarizes the processed dataset after label mapping, validation and speaker-independent splitting.

The analysis focuses on:

- Number of samples in each dataset.
- Number of speakers in each dataset.
- Emotion label distribution.
- Duration distribution.
- Train, validation and test split distribution.

These statistics help identify possible imbalance or bias before model training.

In [ ]:
COMMON_EMOTIONS = ["neutral", "happy", "sad", "angry", "fear", "disgust"]

overview = (
    metadata
    .groupby("dataset")
    .agg(
        samples=("sample_id", "count"),
        speakers=("speaker_id", "nunique"),
        mean_duration_s=("duration", "mean"),
        total_hours=("duration", lambda x: x.sum() / 3600)
    )
    .round(3)
    .reset_index()
)

display(overview)

In [ ]:
emotion_count = (
    metadata["emotion"]
    .value_counts()
    .reindex(COMMON_EMOTIONS)
    .reset_index()
)

emotion_count.columns = ["emotion", "samples"]
display(emotion_count)

split_count = pd.crosstab(metadata["split"], metadata["emotion"]).reindex(
    ["train", "validation", "test"]
)

display(split_count)

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=metadata, x="dataset", order=metadata["dataset"].value_counts().index)
plt.title("Number of Samples by Dataset")
plt.xlabel("Dataset")
plt.ylabel("Number of Samples")
plt.tight_layout()

save_path = FIGURE_DIR / "samples_by_dataset.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(data=metadata, x="emotion", order=COMMON_EMOTIONS)
plt.title("Number of Samples by Emotion")
plt.xlabel("Emotion")
plt.ylabel("Number of Samples")
plt.xticks(rotation=25)
plt.tight_layout()

save_path = FIGURE_DIR / "samples_by_emotion.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=metadata, x="duration", hue="dataset", bins=40, element="step")
plt.title("Audio Duration Distribution by Dataset")
plt.xlabel("Duration (seconds)")
plt.ylabel("Number of Audio Files")
plt.tight_layout()

save_path = FIGURE_DIR / "duration_distribution.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

## Dataset overview analysis

The processed dataset now supports four public speech emotion datasets: **RAVDESS**, **CREMA-D**, **TESS** and **SAVEE**.  
All emotion labels are mapped into six common classes: **neutral, happy, sad, angry, fear and disgust**.  
Labels such as **calm**, **surprise** and **pleasant_surprise** are intentionally excluded so that all datasets share the same target space.

### Dataset distribution

**CREMA-D** remains the largest and most speaker-diverse dataset. **TESS** contributes many clean acted samples but contains only two female speakers. **RAVDESS** adds balanced male/female actors. **SAVEE** is much smaller, with four male speakers, but it is useful because many SER papers use the combined setup RAVDESS + CREMA-D + TESS + SAVEE.

This multi-source design is useful academically, but it also introduces dataset bias. The model may learn recording style, speaker identity or dataset-specific sentence patterns instead of only emotion. For that reason, later training notebooks should report performance by dataset, by emotion and by split.

### Emotion distribution

The six-class setup is easier to compare across datasets than a seven-class setup with surprise. CREMA-D does not consistently provide surprise in the same way as the other datasets, so keeping six shared labels avoids an imbalanced class that comes only from some sources.

### Duration distribution

The duration distribution should be checked carefully after adding SAVEE. SAVEE recordings may have a different length profile and recording condition compared with TESS, CREMA-D and RAVDESS. Fixed-duration preprocessing, silence trimming and feature scaling remain important so that the model focuses on acoustic-emotional patterns rather than raw duration.

### Evaluation concern

The recommended report should distinguish two ideas:

1. **Strict speaker-aware evaluation**: better for real generalization because speakers do not overlap between train/validation/test.
2. **Paper-comparable random evaluation**: useful for comparison with many baseline papers, but easier and more vulnerable to speaker/style leakage.


## 4. Checking extracted baseline features

The baseline feature file contains fixed-length feature vectors extracted from each audio sample.  
These features are mainly designed for traditional machine learning models such as SVM or Random Forest.

The feature vector includes statistical summaries of MFCC, delta MFCC, delta-delta MFCC, RMS energy, zero-crossing rate, spectral centroid and spectral bandwidth.

In [ ]:
X = features["X"]
X_scaled = features["X_scaled"]
y = features["y"]
sample_ids = features["sample_id"]
splits = features["split"]

print("X shape:", X.shape)
print("X_scaled shape:", X_scaled.shape)
print("Number of labels:", len(y))
print("Number of sample ids:", len(sample_ids))
print("Splits:", pd.Series(splits).value_counts().to_dict())
print("Labels:", pd.Series(y).value_counts().to_dict())

## 5. Speech feature visualization

This section visualizes important speech representations used in Speech Emotion Recognition.

The main features include:

- **Waveform:** shows how the audio amplitude changes over time.
- **Spectrogram:** shows the frequency content of the speech signal over time.
- **MFCC:** represents the spectral shape of speech and is widely used in speech processing.
- **Log-Mel spectrogram:** represents audio energy on the Mel scale and is suitable for CNN-based models.

These visualizations help us understand how emotional speech can be represented before model training.

In [ ]:
TARGET_SR = int(config.get("target_sample_rate", 16000))

def load_processed_audio(row):
    """
    Load processed 16 kHz audio from ser_processed/audio_16k.
    This notebook does not depend on raw dataset paths.
    """
    audio_path = row.get("processed_audio_path", None)

    if pd.isna(audio_path) or not Path(audio_path).exists():
        raise FileNotFoundError(
            f"Processed audio not found for sample_id={row['sample_id']}. "
            f"Expected path: {AUDIO_16K_DIR / (row['sample_id'] + '.wav')}"
        )

    y, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True)
    return y, sr

print("Target sample rate:", TARGET_SR)

In [ ]:
metadata["processed_audio_path"] = metadata["sample_id"].apply(
    lambda sid: str(AUDIO_16K_DIR / f"{sid}.wav")
    if (AUDIO_16K_DIR / f"{sid}.wav").exists()
    else np.nan
)

print("Total metadata samples:", len(metadata))
print("Available processed audio:", metadata["processed_audio_path"].notna().sum())
print("Missing processed audio:", metadata["processed_audio_path"].isna().sum())

### Note about processed audio files

The metadata file contains 14,353 samples, but only 10,558 processed 16 kHz audio files are currently available in the `audio_16k` folder.

Therefore, the general dataset overview uses the full metadata, while waveform-based visualization and acoustic analysis are performed only on samples that have available processed audio.

This is acceptable for the current EDA and feature visualization stage because the available subset is still large enough for qualitative analysis.

In [ ]:
metadata_audio = metadata[
    metadata["processed_audio_path"].notna()
    & metadata["emotion"].isin(COMMON_EMOTIONS)
].copy()

print("Samples available for audio visualization:", len(metadata_audio))

print("\nSamples by dataset:")
display(metadata_audio["dataset"].value_counts())

print("\nSamples by emotion:")
display(metadata_audio["emotion"].value_counts().reindex(COMMON_EMOTIONS))

In [ ]:
TARGET_SR = int(config.get("target_sample_rate", 16000))

def load_processed_audio(row):
    """
    Load processed 16 kHz audio from ser_processed/audio_16k.
    This avoids using old raw dataset paths.
    """
    audio_path = row["processed_audio_path"]

    if pd.isna(audio_path) or not Path(audio_path).exists():
        raise FileNotFoundError(f"Processed audio not found: {audio_path}")

    y, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True)
    return y, sr

print("Target sample rate:", TARGET_SR)

### Representative samples for feature visualization

One audio sample is selected from each emotion class.  
These samples are used to visualize waveform, spectrogram, MFCC and Log-Mel spectrogram.

In [ ]:
examples = (
    metadata_audio
    .groupby("emotion", group_keys=False)
    .sample(1, random_state=RANDOM_STATE)
    .sort_values("emotion")
)

display(examples[[
    "sample_id", "dataset", "emotion", "speaker_id",
    "duration", "split", "processed_audio_path"
]])

In [ ]:
def plot_speech_features(row, save=True):
    """
    Plot waveform, spectrogram, MFCC and Log-Mel spectrogram for one audio sample.
    """
    y, sr = load_processed_audio(row)

    stft = np.abs(librosa.stft(y, n_fft=1024, hop_length=256))
    stft_db = librosa.amplitude_to_db(stft, ref=np.max)

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=40,
        n_fft=1024,
        hop_length=256
    )

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=1024,
        hop_length=256,
        n_mels=128,
        fmax=8000
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)

    fig, axes = plt.subplots(4, 1, figsize=(13, 11))

    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"Waveform — {row['emotion']} ({row['dataset']})")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Amplitude")

    img1 = librosa.display.specshow(
        stft_db,
        sr=sr,
        hop_length=256,
        x_axis="time",
        y_axis="hz",
        ax=axes[1]
    )
    axes[1].set_title("Spectrogram")
    fig.colorbar(img1, ax=axes[1], format="%+2.0f dB")

    img2 = librosa.display.specshow(
        mfcc,
        sr=sr,
        hop_length=256,
        x_axis="time",
        ax=axes[2]
    )
    axes[2].set_title("MFCC")
    fig.colorbar(img2, ax=axes[2])

    img3 = librosa.display.specshow(
        log_mel,
        sr=sr,
        hop_length=256,
        x_axis="time",
        y_axis="mel",
        ax=axes[3]
    )
    axes[3].set_title("Log-Mel Spectrogram")
    fig.colorbar(img3, ax=axes[3], format="%+2.0f dB")

    plt.tight_layout()

    if save:
        filename = f"features_{row['emotion']}_{row['sample_id']}.png"
        save_path = FIGURE_DIR / filename
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()

In [ ]:
example_row = examples.iloc[0]
plot_speech_features(example_row)

In [ ]:
for _, row in examples.iterrows():
    plot_speech_features(row)

### Feature visualization analysis

The waveform shows how the amplitude of the speech signal changes over time.  
It helps observe loudness, silence regions and speaking continuity, but it does not provide enough frequency information for emotion classification.

The spectrogram shows how frequency components change over time.  
It is useful because emotional speech may affect the energy distribution across different frequency regions.

MFCC represents the spectral shape of speech in a compact form.  
It is widely used in traditional speech processing and is suitable for baseline models such as SVM or Random Forest.

The Log-Mel spectrogram represents speech energy on the Mel frequency scale.  
Compared with MFCC, it preserves more time-frequency structure, making it more suitable for deep learning models such as CNN or CNN-BiGRU/LSTM.

Overall, these visualizations explain why multiple feature types are useful in Speech Emotion Recognition. MFCC is compact and explainable, while Log-Mel spectrogram provides richer visual patterns for deep learning models.

## 6. Acoustic speaking indicators

Besides emotion classification, this project also analyzes acoustic indicators that can support speaking feedback.

The main acoustic indicators include:

- **RMS energy:** represents the loudness or energy of the voice.
- **Pitch/F0 mean:** represents the average voice pitch.
- **Pitch/F0 standard deviation:** represents pitch variation or voice instability.
- **Zero-crossing rate (ZCR):** represents how frequently the signal changes sign, which may reflect sharpness or noisiness.
- **Pause ratio:** represents the proportion of silence in the audio.
- **Speaking continuity:** represents how continuous the speech is.

These indicators are not used for medical stress diagnosis.  
They are used only to provide feedback for speaking practice, such as voice stability, speaking continuity and possible hesitation.

In [ ]:
from tqdm.auto import tqdm

def safe_stats(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return 0.0, 0.0
    return float(np.mean(x)), float(np.std(x))


def compute_acoustic_indicators(row):
    """
    Compute acoustic speaking indicators from processed 16 kHz audio.
    """
    y, sr = load_processed_audio(row)

    # RMS energy
    rms = librosa.feature.rms(
        y=y,
        frame_length=1024,
        hop_length=256
    )[0]

    # Zero-crossing rate
    zcr = librosa.feature.zero_crossing_rate(
        y,
        frame_length=1024,
        hop_length=256
    )[0]

    # Silence / speech intervals
    intervals = librosa.effects.split(y, top_db=30)

    speech_samples = sum(end - start for start, end in intervals)
    total_samples = max(len(y), 1)

    pause_ratio = 1 - speech_samples / total_samples
    speaking_continuity = 1 - pause_ratio
    pause_count = max(len(intervals) - 1, 0)

    # Pitch / F0
    try:
        f0, voiced_flag, _ = librosa.pyin(
            y,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr
        )

        voiced_f0 = f0[np.isfinite(f0)] if f0 is not None else np.array([])
        pitch_mean, pitch_std = safe_stats(voiced_f0)
        voiced_ratio = float(np.mean(voiced_flag)) if voiced_flag is not None else 0.0

    except Exception:
        pitch_mean, pitch_std, voiced_ratio = 0.0, 0.0, 0.0

    rms_mean, rms_std = safe_stats(rms)
    zcr_mean, zcr_std = safe_stats(zcr)

    return {
        "sample_id": row["sample_id"],
        "dataset": row["dataset"],
        "emotion": row["emotion"],
        "split": row["split"],
        "pitch_mean": pitch_mean,
        "pitch_std": pitch_std,
        "voiced_ratio": voiced_ratio,
        "rms_mean": rms_mean,
        "rms_std": rms_std,
        "zcr_mean": zcr_mean,
        "zcr_std": zcr_std,
        "pause_ratio": float(pause_ratio),
        "pause_count": int(pause_count),
        "speaking_continuity": float(speaking_continuity),
    }

In [ ]:
MAX_PER_EMOTION = 60

indicator_source = (
    metadata_audio
    .groupby("emotion", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), MAX_PER_EMOTION), random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

print("Total samples for acoustic indicator analysis:", len(indicator_source))
display(indicator_source["emotion"].value_counts().reindex(COMMON_EMOTIONS))

In [ ]:
indicator_rows = []

for _, row in tqdm(indicator_source.iterrows(), total=len(indicator_source)):
    try:
        indicator_rows.append(compute_acoustic_indicators(row))
    except Exception as exc:
        print("Skipped:", row["sample_id"], exc)

indicators = pd.DataFrame(indicator_rows)

save_path = OUTPUT_DIR / "acoustic_indicators_minhcuong.csv"
indicators.to_csv(save_path, index=False)

print("Saved:", save_path)
print("Indicator shape:", indicators.shape)

display(indicators.head())

In [ ]:
indicator_summary = (
    indicators
    .groupby("emotion")[[
        "pitch_mean",
        "pitch_std",
        "rms_mean",
        "zcr_mean",
        "pause_ratio",
        "speaking_continuity"
    ]]
    .mean()
    .reindex(COMMON_EMOTIONS)
    .round(4)
)

display(indicator_summary)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

plot_items = [
    ("pitch_mean", "Mean Pitch / F0"),
    ("pitch_std", "Pitch Variation"),
    ("rms_mean", "RMS Energy"),
    ("zcr_mean", "Zero-Crossing Rate"),
    ("pause_ratio", "Pause Ratio"),
    ("speaking_continuity", "Speaking Continuity"),
]

for ax, (col, title) in zip(axes.flat, plot_items):
    sns.boxplot(
        data=indicators,
        x="emotion",
        y=col,
        order=COMMON_EMOTIONS,
        ax=ax,
        showfliers=False
    )
    ax.set_title(title)
    ax.set_xlabel("Emotion")
    ax.set_ylabel(col)
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()

save_path = FIGURE_DIR / "acoustic_indicators_boxplot.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

### Acoustic indicators analysis

The boxplots show that acoustic indicators vary across emotion classes, but the differences should be interpreted carefully because they may also be affected by speaker identity, dataset source and recording condition.

For **mean pitch / F0**, emotions such as **happy, angry and fear** tend to have higher pitch values than neutral, sad and disgust. This suggests that high-arousal emotions may be expressed with a higher voice pitch.

For **pitch variation**, the **happy** class shows the largest variation. This means happy speech may contain more expressive pitch movement. In contrast, neutral speech has lower pitch variation, which is consistent with a more stable speaking style.

For **RMS energy**, the distributions are relatively close across emotions. Some emotions such as fear and happy show slightly higher energy ranges, but the difference is not strong enough to be used alone for emotion classification.

For **zero-crossing rate**, angry speech tends to have a slightly higher range than other emotions. This may indicate sharper or more active signal changes, but it should only be considered as a supporting feature.

For **pause ratio** and **speaking continuity**, most emotions have very low pause ratio and high continuity. However, angry and disgust show more variation in pause ratio. This means some samples in these classes contain more silence or discontinuity.

Overall, these indicators are useful for speaking feedback because they can describe voice energy, pitch stability and speaking continuity. However, they should not be used as medical stress indicators. In this project, they are only used to support feedback for speaking practice.

## 7. Emotion timeline framework

The emotion timeline is designed to analyze a longer speech recording by dividing it into short segments.

In the final system, each segment will be passed into the trained Speech Emotion Recognition model to predict an emotion label.  
Then, the predicted labels are plotted over time to show how the speaker's emotional expression changes during the speech.

At this stage, the notebook builds the timeline framework and extracts segment-level acoustic indicators.  
The actual model prediction can be added later after the SER model is trained.

In [ ]:
def segment_audio(y, sr, segment_duration=3.0, hop_duration=3.0):
    """
    Split audio into fixed-length segments.
    """
    segment_length = int(segment_duration * sr)
    hop_length = int(hop_duration * sr)

    segments = []

    for start in range(0, len(y), hop_length):
        end = start + segment_length
        segment = y[start:end]

        if len(segment) < int(0.5 * sr):
            continue

        if len(segment) < segment_length:
            segment = np.pad(segment, (0, segment_length - len(segment)))

        segments.append({
            "start_time": start / sr,
            "end_time": min(end, len(y)) / sr,
            "audio": segment
        })

    return segments

In [ ]:
def compute_segment_indicators(segment_audio_array, sr):
    """
    Compute acoustic indicators for one audio segment.
    """
    y = segment_audio_array

    rms = librosa.feature.rms(y=y, frame_length=1024, hop_length=256)[0]
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=1024, hop_length=256)[0]

    intervals = librosa.effects.split(y, top_db=30)
    speech_samples = sum(end - start for start, end in intervals)
    pause_ratio = 1 - speech_samples / max(len(y), 1)
    speaking_continuity = 1 - pause_ratio

    try:
        f0, voiced_flag, _ = librosa.pyin(
            y,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr
        )
        voiced_f0 = f0[np.isfinite(f0)] if f0 is not None else np.array([])
        pitch_mean, pitch_std = safe_stats(voiced_f0)
    except Exception:
        pitch_mean, pitch_std = 0.0, 0.0

    rms_mean, rms_std = safe_stats(rms)
    zcr_mean, zcr_std = safe_stats(zcr)

    return {
        "pitch_mean": pitch_mean,
        "pitch_std": pitch_std,
        "rms_mean": rms_mean,
        "rms_std": rms_std,
        "zcr_mean": zcr_mean,
        "zcr_std": zcr_std,
        "pause_ratio": float(pause_ratio),
        "speaking_continuity": float(speaking_continuity)
    }

In [ ]:
timeline_sample = metadata_audio.sample(1, random_state=RANDOM_STATE).iloc[0]

print("Selected sample:")
display(timeline_sample[["sample_id", "dataset", "emotion", "duration", "processed_audio_path"]])

y_timeline, sr_timeline = load_processed_audio(timeline_sample)

print("Audio length:", round(len(y_timeline) / sr_timeline, 2), "seconds")

In [ ]:
segments = segment_audio(
    y_timeline,
    sr_timeline,
    segment_duration=1.0,
    hop_duration=1.0
)

timeline_rows = []

for i, seg in enumerate(segments):
    indicators_seg = compute_segment_indicators(seg["audio"], sr_timeline)

    timeline_rows.append({
        "segment_id": i,
        "start_time": seg["start_time"],
        "end_time": seg["end_time"],
        "true_emotion": timeline_sample["emotion"],
        **indicators_seg
    })

timeline_df = pd.DataFrame(timeline_rows)

display(timeline_df)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(timeline_df["start_time"], timeline_df["rms_mean"], marker="o", label="RMS Energy")
plt.plot(timeline_df["start_time"], timeline_df["pause_ratio"], marker="o", label="Pause Ratio")
plt.plot(timeline_df["start_time"], timeline_df["speaking_continuity"], marker="o", label="Speaking Continuity")

plt.title("Segment-level Acoustic Timeline")
plt.xlabel("Time (seconds)")
plt.ylabel("Indicator Value")
plt.legend()
plt.tight_layout()

save_path = FIGURE_DIR / "segment_acoustic_timeline_demo.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

## 8. Demonstration of emotion timeline

Most samples in RAVDESS, CREMA-D, TESS and SAVEE are short utterances, so a single audio file may contain only a few segments.  
To demonstrate the emotion timeline more clearly, this section creates a longer demo audio by concatenating one sample from each emotion class.

This is not a real model prediction yet.  
The emotion labels in this demo are taken from the known dataset labels and are used only to demonstrate how the final timeline will work.

After the SER model is trained, these known labels can be replaced by predicted labels for each segment.

In [ ]:
import soundfile as sf

def create_concatenated_demo_audio(examples_df, silence_duration=0.3):
    """
    Create a longer demo audio by concatenating one sample from each emotion.
    """
    demo_audio_parts = []
    emotion_ranges = []
    current_time = 0.0

    silence = np.zeros(int(silence_duration * TARGET_SR), dtype=np.float32)

    for _, row in examples_df.iterrows():
        y, sr = load_processed_audio(row)

        start_time = current_time
        end_time = current_time + len(y) / sr

        demo_audio_parts.append(y)
        demo_audio_parts.append(silence)

        emotion_ranges.append({
            "emotion": row["emotion"],
            "dataset": row["dataset"],
            "sample_id": row["sample_id"],
            "start_time": start_time,
            "end_time": end_time
        })

        current_time = end_time + silence_duration

    demo_audio = np.concatenate(demo_audio_parts).astype(np.float32)
    emotion_ranges = pd.DataFrame(emotion_ranges)

    return demo_audio, emotion_ranges


demo_audio, emotion_ranges = create_concatenated_demo_audio(examples)

demo_audio_path = OUTPUT_DIR / "demo_concatenated_emotions.wav"
sf.write(str(demo_audio_path), demo_audio, TARGET_SR)

print("Demo audio saved:", demo_audio_path)
print("Demo audio duration:", round(len(demo_audio) / TARGET_SR, 2), "seconds")

display(emotion_ranges)

In [ ]:
demo_segments = segment_audio(
    demo_audio,
    TARGET_SR,
    segment_duration=1.0,
    hop_duration=1.0
)

print("Number of demo segments:", len(demo_segments))

In [ ]:
def get_emotion_at_time(time_point, emotion_ranges):
    """
    Return the emotion label at a specific time point.
    """
    matched = emotion_ranges[
        (emotion_ranges["start_time"] <= time_point)
        & (emotion_ranges["end_time"] >= time_point)
    ]

    if len(matched) == 0:
        return "silence"

    return matched.iloc[0]["emotion"]


demo_timeline_rows = []

for i, seg in enumerate(demo_segments):
    center_time = (seg["start_time"] + seg["end_time"]) / 2
    emotion_label = get_emotion_at_time(center_time, emotion_ranges)
    seg_indicators = compute_segment_indicators(seg["audio"], TARGET_SR)

    demo_timeline_rows.append({
        "segment_id": i,
        "start_time": seg["start_time"],
        "end_time": seg["end_time"],
        "center_time": center_time,
        "timeline_emotion": emotion_label,
        **seg_indicators
    })

demo_timeline_df = pd.DataFrame(demo_timeline_rows)

display(demo_timeline_df.head(10))
display(demo_timeline_df["timeline_emotion"].value_counts())

In [ ]:
timeline_emotions = COMMON_EMOTIONS + ["silence"]
emotion_to_id = {emotion: i for i, emotion in enumerate(timeline_emotions)}

demo_timeline_df["emotion_id"] = demo_timeline_df["timeline_emotion"].map(emotion_to_id)

plt.figure(figsize=(13, 5))

plt.step(
    demo_timeline_df["start_time"],
    demo_timeline_df["emotion_id"],
    where="post",
    marker="o"
)

plt.yticks(
    list(emotion_to_id.values()),
    list(emotion_to_id.keys())
)

plt.title("Demo Emotion Timeline from Concatenated Speech Segments")
plt.xlabel("Time (seconds)")
plt.ylabel("Emotion")
plt.grid(True)
plt.tight_layout()

save_path = FIGURE_DIR / "demo_emotion_timeline.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

In [ ]:
plt.figure(figsize=(13, 5))

plt.plot(
    demo_timeline_df["start_time"],
    demo_timeline_df["rms_mean"],
    marker="o",
    label="RMS Energy"
)

plt.plot(
    demo_timeline_df["start_time"],
    demo_timeline_df["pause_ratio"],
    marker="o",
    label="Pause Ratio"
)

plt.plot(
    demo_timeline_df["start_time"],
    demo_timeline_df["speaking_continuity"],
    marker="o",
    label="Speaking Continuity"
)

plt.title("Demo Segment-level Acoustic Timeline")
plt.xlabel("Time (seconds)")
plt.ylabel("Indicator Value")
plt.legend()
plt.grid(True)
plt.tight_layout()

save_path = FIGURE_DIR / "demo_acoustic_timeline.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

### Emotion timeline demonstration analysis

The demo emotion timeline shows how a longer speech recording can be divided into short time segments and represented as an emotion sequence over time.

In this demonstration, one sample from each emotion class is concatenated to create a longer audio sequence. The `silence` segments appear because short silent gaps were inserted between different emotion samples. At this stage, the emotion labels are taken from the known dataset labels, so this figure is used only to demonstrate the timeline framework, not real model prediction.

The acoustic timeline shows how **RMS energy**, **pause ratio** and **speaking continuity** change across segments. When pause ratio increases, speaking continuity decreases. This helps the system identify parts of the speech that contain more silence or less continuous speaking.

Overall, this framework shows how the final system can analyze a long speaking recording segment by segment. After the SER model is trained, the known labels in this demo can be replaced by predicted emotion labels to create a real emotion timeline for user speech.

## 9. Rule-based speaking feedback

After extracting acoustic indicators, the system can generate simple speaking feedback.

At this stage, the feedback is rule-based and uses acoustic indicators such as:

- RMS energy for voice strength.
- Pitch standard deviation for pitch stability.
- Pause ratio for hesitation or silence.
- Speaking continuity for fluency.

This feedback is not a medical diagnosis.  
It is only designed to support speaking practice by giving comments about voice energy, stability and continuity.

In [ ]:
def generate_segment_feedback(row):
    """
    Generate simple feedback for one speech segment based on acoustic indicators.
    """
    comments = []

    # Voice energy
    if row["rms_mean"] < 0.05:
        comments.append("Voice energy is low; the speaker may need to speak more clearly or loudly.")
    elif row["rms_mean"] > 0.16:
        comments.append("Voice energy is strong and clear.")
    else:
        comments.append("Voice energy is moderate.")

    # Pause and continuity
    if row["pause_ratio"] > 0.20:
        comments.append("This segment contains noticeable silence or hesitation.")
    elif row["speaking_continuity"] > 0.90:
        comments.append("Speech continuity is good.")
    else:
        comments.append("Speech continuity is acceptable but could be smoother.")

    # Pitch stability
    if row["pitch_std"] > 70:
        comments.append("Pitch variation is high, which may indicate expressive or unstable speaking.")
    elif row["pitch_std"] < 20:
        comments.append("Pitch is relatively stable.")
    else:
        comments.append("Pitch variation is moderate.")

    return " ".join(comments)

In [ ]:
demo_timeline_df["segment_feedback"] = demo_timeline_df.apply(
    generate_segment_feedback,
    axis=1
)

display(demo_timeline_df[[
    "segment_id",
    "start_time",
    "end_time",
    "timeline_emotion",
    "rms_mean",
    "pitch_std",
    "pause_ratio",
    "speaking_continuity",
    "segment_feedback"
]])

In [ ]:
def generate_overall_feedback(timeline_df):
    """
    Generate overall speaking feedback from segment-level indicators.
    """
    avg_rms = timeline_df["rms_mean"].mean()
    avg_pause = timeline_df["pause_ratio"].mean()
    avg_continuity = timeline_df["speaking_continuity"].mean()
    avg_pitch_std = timeline_df["pitch_std"].mean()

    feedback = []

    feedback.append("Overall speaking feedback:")

    if avg_rms < 0.05:
        feedback.append("- The overall voice energy is quite low. The speaker should try to speak more clearly and confidently.")
    elif avg_rms > 0.16:
        feedback.append("- The overall voice energy is strong.")
    else:
        feedback.append("- The overall voice energy is moderate and acceptable.")

    if avg_pause > 0.20:
        feedback.append("- The speech contains several silent or hesitant regions. Reducing long pauses may improve fluency.")
    else:
        feedback.append("- The pause ratio is low, which suggests good speaking continuity.")

    if avg_continuity > 0.90:
        feedback.append("- Speaking continuity is good across most segments.")
    else:
        feedback.append("- Speaking continuity could be improved in some segments.")

    if avg_pitch_std > 70:
        feedback.append("- Pitch variation is high. This can make the speech expressive, but too much variation may sound unstable.")
    elif avg_pitch_std < 20:
        feedback.append("- Pitch is stable, but the speech may sound less expressive.")
    else:
        feedback.append("- Pitch variation is moderate.")

    return "\n".join(feedback)


overall_feedback = generate_overall_feedback(demo_timeline_df)
print(overall_feedback)

In [ ]:
feedback_csv_path = OUTPUT_DIR / "demo_timeline_feedback_minhcuong.csv"
feedback_txt_path = OUTPUT_DIR / "demo_overall_feedback_minhcuong.txt"

demo_timeline_df.to_csv(feedback_csv_path, index=False)

with open(feedback_txt_path, "w", encoding="utf-8") as f:
    f.write(overall_feedback)

print("Saved CSV:", feedback_csv_path)
print("Saved TXT:", feedback_txt_path)

### Speaking feedback analysis

The rule-based feedback module converts acoustic indicators into simple comments for speaking practice.

For each segment, the system checks voice energy, pause ratio, speaking continuity and pitch variation.  
Based on these values, it can generate comments such as low voice energy, good continuity, noticeable silence or high pitch variation.

The overall feedback summarizes the average speaking behavior across all segments.  
This is useful for a speaking practice system because users can receive both segment-level feedback and general feedback for the whole recording.

At this stage, the feedback rules are simple and interpretable.  
In the final version, these rules can be improved by combining acoustic indicators with the predicted emotion timeline from the trained SER model.

## 10. Generated artifacts

This notebook generated several figures and output files for EDA, feature visualization, acoustic indicator analysis and timeline demonstration.

### Figures

The generated figures include:

- Dataset distribution by source.
- Emotion label distribution.
- Audio duration distribution.
- Waveform, spectrogram, MFCC and Log-Mel visualization for six emotion classes.
- Acoustic indicator boxplots.
- Segment-level acoustic timeline.
- Demo emotion timeline.
- Demo acoustic timeline.

These figures can be used directly in the midterm report and presentation slides.

### Output files

The notebook also exported:

- `acoustic_indicators_minhcuong.csv`: acoustic indicators computed from selected samples.
- `demo_concatenated_emotions.wav`: demo audio created by concatenating samples from different emotion classes.
- `demo_timeline_feedback_minhcuong.csv`: segment-level feedback table.
- `demo_overall_feedback_minhcuong.txt`: overall speaking feedback generated from acoustic indicators.

These artifacts demonstrate how the proposed speaking feedback system can analyze emotion-related speech features and generate interpretable feedback.

In [ ]:
print("Figures saved in:", FIGURE_DIR)
for p in sorted(FIGURE_DIR.glob("*.png")):
    print("-", p.name)

print("\nMain artifacts saved in:", OUTPUT_DIR)
for p in [
    OUTPUT_DIR / "acoustic_indicators_minhcuong.csv",
    OUTPUT_DIR / "demo_concatenated_emotions.wav",
    OUTPUT_DIR / "demo_timeline_feedback_minhcuong.csv",
    OUTPUT_DIR / "demo_overall_feedback_minhcuong.txt"
]:
    print("-", p.name, "| exists:", p.exists())

## 11. Rubric checklist

This notebook supports the midterm rubric, especially the Speech Data and Feature Extraction section.

### Rubric 2.1 — Dataset overview

The notebook summarizes the processed dataset from RAVDESS, CREMA-D, TESS and SAVEE.  
It presents the number of samples, dataset sources, emotion labels and general data structure.

### Rubric 2.2 — Dataset structure

The notebook analyzes metadata fields such as dataset name, emotion label, speaker ID, duration and split.  
It also shows the distribution of samples by dataset and emotion class.

### Rubric 2.3 — Data preparation

This notebook continues from the previous preprocessing stage, which included resampling to 16 kHz, label mapping, audio normalization and speaker-independent splitting.  
The notebook uses processed 16 kHz audio files for visualization and acoustic analysis.

### Rubric 2.4 — Speech features

The notebook explains and visualizes important speech features, including waveform, spectrogram, MFCC, Log-Mel spectrogram, RMS energy, pitch/F0, ZCR, pause ratio and speaking continuity.

### Rubric 2.5 — Feature illustration and analysis

The notebook provides charts and figures for dataset distribution, feature visualization, acoustic indicators and segment-level timeline analysis.  
These figures help explain how speech features can support emotion recognition and speaking feedback.

### Initial system demonstration

The notebook also builds an initial timeline and feedback framework.  
Although the final SER model prediction is not included yet, the current implementation demonstrates how long speech can be segmented and analyzed over time.

## 12. Conclusion

This notebook extends the processed speech emotion dataset into a deeper analysis stage for the four-dataset setup: RAVDESS, CREMA-D, TESS and SAVEE.

First, the dataset distribution was analyzed to understand sample imbalance, emotion distribution and duration differences across datasets. Then, several speech feature representations were visualized, including waveform, spectrogram, MFCC and Log-Mel spectrogram.

Next, acoustic speaking indicators were extracted and analyzed. RMS energy, pitch/F0, ZCR, pause ratio and speaking continuity provide useful information for speaking feedback. These indicators can describe voice strength, pitch stability, hesitation and fluency.

Finally, the notebook built a demonstration framework for segment-level analysis. A longer demo audio was created by concatenating short emotion samples, then the audio was divided into segments to create an emotion timeline and acoustic timeline. A simple rule-based feedback module was also implemented to generate speaking feedback from acoustic indicators.

Overall, this notebook provides the EDA foundation for the baseline training notebook. After adding SAVEE, the report should explicitly discuss dataset bias, speaker imbalance and the difference between strict speaker-aware evaluation and random paper-comparable evaluation.
